**Gradient Boosting (often called GBD or GBDT - Gradient Boosted Decision Trees)** is an advanced sequential ensemble technique.

# Gradient Boosting Regressor

## The Core Intuition

1. **The Base Guess:** The algorithm starts by making a single, baseline prediction for the entire dataset (usually just the average value of your target variable).

$$F_0(x) = \bar{y}$$

2. **Calculate the Error:** It calculates how far off this base guess is for every single row. This difference (Actual Value minus Predicted Value) is called a **Residual.**

$$r_i = y_i - F_{m-1}(x_i)$$

**Note:** In Gradient Boosting Regressor we use Half Squarred Error Loss. The derivative of which becomes y - F.

3. **Train on Errors:** The next decision tree is not trained to predict the final target. It is trained specifically to predict those residuals (the errors).

$$h_m(x) = \begin{cases} \text{mean}(r \text{ where } x \le T) & \text{if } x \le T \\ \text{mean}(r \text{ where } x > T) & \text{if } x > T \end{cases}$$

4. **Update the Prediction:** The algorithm adds the new tree's prediction to the running total, bringing the overall guess closer to the true value.

$$F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$$

5. **Repeat:** The process repeats for many rounds. Each new tree isolates and chips away at the remaining errors left behind by the combined group of previous trees.

$$F_M(x) = \bar{y} + \nu \cdot h_1(x) + \nu \cdot h_2(x) + \dots + \nu \cdot h_M(x)$$

## Mathematical Example

1. **The Dataset**
    We want to predict a house price (Y) based on its size (X).
    
    |Row|Feature (X)|True Target (Y)|
    |---|---|---|
    |1|1.0|10|
    |2|2.0|20|
    |3|3.0|60|

    **Base Guess** (Mean value of Y): For squared error loss, this base guess is simply the average value of Y:
    $$F_{0}(x)=\frac{10+20+60}{3}=\frac{90}{3}=\mathbf{30}$$
    $$\hat y = [30,30,30]$$

2. **Calcuate the Error**:
    $$\text{Residual} = Y - F₀(x)$$
    - Row 1 Residual : 10 - 30  = -20
    - Row 2 Residual : 20 - 30  = -10
    - Row 3 Residual : 60 - 30  = +30

3. **Train Tree to Predict the Residuals**:
    We now train a decision tree stump (h₁(x)). Crucially, the target column for this tree is the Residual column, not the original Y.
    |Row|Feature (X)|Target for Tree 1 (Residual)|
    |---|---|---|
    |1|1.0|-20|
    |2|2.0|-10|
    |3|3.0|+30|

    Let's say the tree selects a split threshold at X ≤ 2.5:
    - Left Branch: Contains Row 1 and Row 2. The tree predicts the average of their residuals: $\frac{-20 + (-10)}{2} = \mathbf{-15}$
    - Right Branch (X > 2.5): Contains Row 3. The tree predicts its residual: $\mathbf{+30}$.

    Tree 1 Predictions $(h_1(x))$: [-15,-15,+30]

4. **Update the Running Ensemble Prediction (F₁(x))**
    We update our overall prediction by adding the new tree's output to our previous guess.
    To prevent overfitting, we scale the new tree's contribution by our Learning Rate (ν = 0.1):
    $$F_{1}(x)=F_{0}(x)+(\nu \cdot h_{1}(x))$$

    - Row 1 New Guess: $30 + (0.1 \cdot (-15)) = 30 - 1.5 = \mathbf{28.5}$ (Brought closer to 10)
    - Row 2 New Guess: $30 + (0.1 \cdot (-15)) = 30 - 1.5 = \mathbf{28.5}$ (Brought closer to 20)
    - Row 3 New Guess: $30 + (0.1 \cdot (+30)) = 30 + 3.0 = \mathbf{33.0}$ (Brought closer to 60)

5. **Calculate the Next Residuals for the next round**
    Round 1 is complete. To prepare for Round 2, we calculate a brand-new set of residuals based on our updated guesses: New Residual = Y - F₁(x).
    - Row 1 New Residual: $10 - 28.5 = \mathbf{-18.5}$
    - Row 2 New Residual: $20 - 28.5 = \mathbf{-8.5}$
    - Row 3 New Residual: $60 - 33.0 = \mathbf{+27.0}$

    Now Repeat the same steps again ....


6. **Prediction**
    When a brand-new, completely unknown test sample arrives, it does not have a \(Y\) value, so we cannot calculate residuals for it. Instead, we pass its features through the exact chain of components we built:
    $$\text{Final\ Prediction}=\text{Base\ Mean}+\nu \cdot \text{Tree}_{1}(x)+\nu \cdot \text{Tree}_{2}(x)+\dots +\nu \cdot \text{Tree}_{M}(x)$$





## Python Code

In [9]:
import numpy as np


class SimpleDecisionStump:
    """Decision stump built specifically to predict mean residuals."""

    def __init__(self):
        self.feature_idx = 0
        self.threshold = None
        self.left_mean = 0.0
        self.right_mean = 0.0

    def fit(self, X, residuals):
        n_samples, n_features = X.shape
        best_sse = float("inf")

        for f_idx in range(n_features):
            X_col = X[:, f_idx]
            sorted_unique = np.sort(np.unique(X_col))

            if len(sorted_unique) <= 1:
                continue

            # Candidate midpoint thresholds
            thresholds = (sorted_unique[:-1] + sorted_unique[1:]) / 2.0

            for t in thresholds:
                left_mask = X_col <= t
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                # Mean of residuals in each branch
                left_val = np.mean(residuals[left_mask])
                right_val = np.mean(residuals[right_mask])

                # Calculate Sum of Squared Errors (SSE)
                preds = np.where(left_mask, left_val, right_val)
                sse = np.sum((residuals - preds) ** 2)

                if sse < best_sse:
                    best_sse = sse
                    self.feature_idx = f_idx
                    self.threshold = t
                    self.left_mean = left_val
                    self.right_mean = right_val

    def predict(self, X):
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_mean,
            self.right_mean,
        )


class SimpleGradientBoostingRegressor:
    """GBDT using simplified MSE residual rules."""

    def __init__(self, n_estimators=3, learning_rate=0.1):
        self.M = n_estimators
        self.nu = learning_rate
        self.y_bar = 0.0
        self.trees = []

    def fit(self, X, y):
        # Step 1: Base guess is the mean of y
        self.y_bar = np.mean(y)
        F = np.full_like(y, self.y_bar, dtype=np.float64)

        print("--- GBDT TRAINING LOOP ---")
        print(f"Base Guess F0(x) = {self.y_bar:.3f}\n")

        for m in range(self.M):
            # Step 2: Compute Residuals (r = y - F)
            residuals = y - F

            # Step 3: Train Tree Stump on Residuals
            tree = SimpleDecisionStump()
            tree.fit(X, residuals)

            # Step 4: Tree prediction h_m(x)
            h_m = tree.predict(X)

            # Step 5: Update F_m(x) = F_{m-1}(x) + nu * h_m(x)
            F = F + self.nu * h_m

            self.trees.append(tree)

            print(
                f"Round {m+1}: Split @ X[{tree.feature_idx}] <= {tree.threshold:.2f}"
            )
            print(f"   Left Mean = {tree.left_mean:.3f} | Right Mean = {tree.right_mean:.3f}")
            print(f"   Current Predictions F_{m+1}(x): {np.round(F, 3)}")

    def predict(self, X_new):
        # Step 6: Final prediction = y_bar + nu * tree1(x) + nu * tree2(x) ...
        predictions = np.full(X_new.shape[0], self.y_bar, dtype=np.float64)

        for tree in self.trees:
            predictions += self.nu * tree.predict(X_new)

        return predictions


if __name__ == "__main__":
    X_train = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
    y_train = np.array([2.0, 5.0, 7.0, 4.0, 8.0])

    model = SimpleGradientBoostingRegressor(n_estimators=5, learning_rate=0.1)
    model.fit(X_train, y_train)

    X_test = np.array([[2.5], [4.2]])
    preds = model.predict(X_test)

    print("\n--- INFERENCE RESULTS ---")
    for x_val, pred in zip(X_test.ravel(), preds):
        print(f"Final Prediction for X = {x_val}: {pred:.3f}")

--- GBDT TRAINING LOOP ---
Base Guess F0(x) = 5.200

Round 1: Split @ X[0] <= 1.50
   Left Mean = -3.200 | Right Mean = 0.800
   Current Predictions F_1(x): [4.88 5.28 5.28 5.28 5.28]
Round 2: Split @ X[0] <= 1.50
   Left Mean = -2.880 | Right Mean = 0.720
   Current Predictions F_2(x): [4.592 5.352 5.352 5.352 5.352]
Round 3: Split @ X[0] <= 4.50
   Left Mean = -0.662 | Right Mean = 2.648
   Current Predictions F_3(x): [4.526 5.286 5.286 5.286 5.617]
Round 4: Split @ X[0] <= 1.50
   Left Mean = -2.526 | Right Mean = 0.631
   Current Predictions F_4(x): [4.273 5.349 5.349 5.349 5.68 ]
Round 5: Split @ X[0] <= 4.50
   Left Mean = -0.580 | Right Mean = 2.320
   Current Predictions F_5(x): [4.215 5.291 5.291 5.291 5.912]

--- INFERENCE RESULTS ---
Final Prediction for X = 2.5: 5.291
Final Prediction for X = 4.2: 5.291
